# Week 2 Assignment: Sales Data Analysis using SQL

**Objective:** Analyze sales data using SQL with filtering, aggregation, and business queries.

**Steps:**
1. Load dataset into a SQL database (SQLite).
2. Explore table (schema, sample data).
3. Apply WHERE filters (region, category, date, sales).
4. Use GROUP BY for aggregations (sales, quantity, averages).
5. Sort and limit results (top products, top categories).
6. Solve use cases (monthly trends, top customers, duplicates).
7. Validate results (row counts, data quality).

*Note: This notebook uses SQLite and Pandas to execute SQL queries directly in Python.*

In [1]:
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

### 1. Load Dataset into SQL Database

In [2]:
file_path = 'Sample - Superstore.csv'

if os.path.exists(file_path):
    # Using windows-1252 encoding since Kaggle CSVs often use it
    df = pd.read_csv(file_path, encoding='windows-1252')
    print("Dataset loaded from CSV successfully.")
else:
    print("Dataset not found! Creating mock data for demonstration...")
    data = {
        'Order ID': ['CA-2023-1', 'CA-2023-2', 'CA-2023-3', 'US-2023-4', 'CA-2023-5'],
        'Order Date': ['2023-01-15', '2023-01-20', '2023-02-10', '2023-02-15', '2023-03-05'],
        'Customer Name': ['John Doe', 'Jane Smith', 'John Doe', 'Alice Brown', 'Bob White'],
        'Region': ['East', 'West', 'East', 'Central', 'West'],
        'Category': ['Furniture', 'Technology', 'Office Supplies', 'Technology', 'Furniture'],
        'Product Name': ['Chair', 'Phone', 'Paper', 'Laptop', 'Table'],
        'Sales': [250.0, 800.0, 15.0, 1200.0, 300.0],
        'Quantity': [2, 1, 3, 1, 4]
    }
    df = pd.DataFrame(data)

# Connect to SQLite database (creates one in memory)
conn = sqlite3.connect(':memory:')

# Clean column names (replace spaces with underscores for easier SQL querying)
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# Load data into SQLite table named 'sales_data'
df.to_sql('sales_data', conn, index=False, if_exists='replace')
print("Data successfully loaded into SQLite database table 'sales_data'.")

Dataset not found! Creating mock data for demonstration...
Data successfully loaded into SQLite database table 'sales_data'.


### 2. Explore Table (Schema & Sample Data)

In [3]:
# Define a helper function to run SQL queries and display as pandas dataframe
def run_query(query):
    return pd.read_sql_query(query, conn)

print("--- TABLE SCHEMA ---")
schema_query = "PRAGMA table_info(sales_data);"
display(run_query(schema_query))

print("\n--- SAMPLE DATA ---")
sample_query = "SELECT * FROM sales_data LIMIT 5;"
display(run_query(sample_query))

--- TABLE SCHEMA ---


,cid,name,type,notnull,dflt_value,pk
0,0,Order_ID,TEXT,0,None,0
1,1,Order_Date,TEXT,0,None,0
2,2,Customer_Name,TEXT,0,None,0
3,3,Region,TEXT,0,None,0
4,4,Category,TEXT,0,None,0
5,5,Product_Name,TEXT,0,None,0
6,6,Sales,REAL,0,None,0
7,7,Quantity,INTEGER,0,None,0



--- SAMPLE DATA ---


,Order_ID,Order_Date,Customer_Name,Region,Category,Product_Name,Sales,Quantity
0,CA-2023-1,2023-01-15,John Doe,East,Furniture,Chair,250.0,2
1,CA-2023-2,2023-01-20,Jane Smith,West,Technology,Phone,800.0,1
2,CA-2023-3,2023-02-10,John Doe,East,Office Supplies,Paper,15.0,3
3,US-2023-4,2023-02-15,Alice Brown,Central,Technology,Laptop,1200.0,1
4,CA-2023-5,2023-03-05,Bob White,West,Furniture,Table,300.0,4


### 3. Apply WHERE Filters (Region, Category, Sales)

In [4]:
print("--- FILTER: Region='West' AND Sales > 100 ---")
filter_query = """
SELECT Order_ID, Customer_Name, Region, Category, Sales 
FROM sales_data 
WHERE Region = 'West' AND Sales > 100
LIMIT 5;
"""
display(run_query(filter_query))

--- FILTER: Region='West' AND Sales > 100 ---


,Order_ID,Customer_Name,Region,Category,Sales
0,CA-2023-2,Jane Smith,West,Technology,800.0
1,CA-2023-5,Bob White,West,Furniture,300.0


### 4. Use GROUP BY for Aggregations

In [5]:
print("--- AGGREGATION: Total Sales and Average Quantity by Category ---")
agg_query = """
SELECT 
    Category,
    SUM(Sales) as Total_Sales,
    AVG(Quantity) as Avg_Quantity,
    COUNT(*) as Total_Orders
FROM sales_data
GROUP BY Category;
"""
display(run_query(agg_query))

--- AGGREGATION: Total Sales and Average Quantity by Category ---


,Category,Total_Sales,Avg_Quantity,Total_Orders
0,Furniture,550.0,3.0,2
1,Office Supplies,15.0,3.0,1
2,Technology,2000.0,1.0,2


### 5. Sort and Limit Results (Top Products)

In [6]:
print("--- TOP 3 PRODUCTS BY SALES ---")
top_products_query = """
SELECT Product_Name, Category, SUM(Sales) as Total_Sales
FROM sales_data
GROUP BY Product_Name
ORDER BY Total_Sales DESC
LIMIT 3;
"""
display(run_query(top_products_query))

--- TOP 3 PRODUCTS BY SALES ---


,Product_Name,Category,Total_Sales
0,Laptop,Technology,1200.0
1,Phone,Technology,800.0
2,Table,Furniture,300.0


### 6. Solve Business Use Cases

In [7]:
print("--- MONTHLY TRENDS (Based on Order Date) ---")
# SQLite uses SUBSTR to extract year-month from YYYY-MM-DD strings
trend_query = """
SELECT 
    SUBSTR(Order_Date, 1, 7) as Order_Month, 
    SUM(Sales) as Monthly_Sales
FROM sales_data
GROUP BY Order_Month
ORDER BY Order_Month;
"""
display(run_query(trend_query))

print("\n--- TOP CUSTOMERS BY LIFETIME VALUE ---")
top_customers_query = """
SELECT Customer_Name, SUM(Sales) as Total_Spent, COUNT(Order_ID) as Total_Orders
FROM sales_data
GROUP BY Customer_Name
ORDER BY Total_Spent DESC
LIMIT 3;
"""
display(run_query(top_customers_query))

print("\n--- FIND DUPLICATE ORDERS (if any) ---")
duplicates_query = """
SELECT Order_ID, COUNT(*) as Occurrences
FROM sales_data
GROUP BY Order_ID
HAVING Occurrences > 1;
"""
display(run_query(duplicates_query))

--- MONTHLY TRENDS (Based on Order Date) ---


,Order_Month,Monthly_Sales
0,2023-01,1050.0
1,2023-02,1215.0
2,2023-03,300.0



--- TOP CUSTOMERS BY LIFETIME VALUE ---


,Customer_Name,Total_Spent,Total_Orders
0,Alice Brown,1200.0,1
1,Jane Smith,800.0,1
2,Bob White,300.0,1



--- FIND DUPLICATE ORDERS (if any) ---


,Order_ID,Occurrences


### 7. Validate Results

In [8]:
print("--- DATA VALIDATION: Total Row Count ---")
row_count_query = "SELECT COUNT(*) as Total_Rows FROM sales_data;"
display(run_query(row_count_query))

print("\n--- DATA VALIDATION: Checking for NULL Sales ---")
null_sales_query = "SELECT COUNT(*) as Missing_Sales FROM sales_data WHERE Sales IS NULL;"
display(run_query(null_sales_query))

# Close connection
conn.close()

--- DATA VALIDATION: Total Row Count ---


,Total_Rows
0,5



--- DATA VALIDATION: Checking for NULL Sales ---


,Missing_Sales
0,0
